# SST-2 Order-Sensitivity Report View

This notebook is for the writeup stage. It assumes the training sweep and analysis have already been run, and focuses on:

- loading the finished analysis outputs
- showing high-signal tables
- surfacing the main quantitative findings
- displaying figures ready for the report


In [ ]:
import json
import os
import pathlib
import subprocess

import pandas as pd
from IPython.display import Image, Markdown, display

CWD = pathlib.Path.cwd()
REPO_URL = 'https://github.com/tarang-tp/ur2phd'
REPO_DIRNAME = 'ur2phd'
USE_DRIVE = True
DRIVE_ANALYSIS_DIR = pathlib.Path('/content/drive/MyDrive/ur2phd_colab/analysis')
required_analysis_files = [
    'summary.json',
    'model_scores.csv',
    'sentence_summary.csv',
    'stable_vs_unstable_comparison.csv',
    'top_unstable_sentences.csv',
]

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ANALYSIS_DIR = DRIVE_ANALYSIS_DIR
else:
    clone_target = CWD / REPO_DIRNAME
    if not clone_target.exists():
        print('Project checkout not found locally. Cloning from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(clone_target)], check=True)
    else:
        print('Existing project checkout found. Pulling latest changes...')
        subprocess.run(['git', '-C', str(clone_target), 'pull', '--ff-only'], check=True)
    os.chdir(clone_target)
    ANALYSIS_DIR = clone_target / 'analysis'

missing = [name for name in required_analysis_files if not (ANALYSIS_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        'Missing analysis outputs in ' + str(ANALYSIS_DIR) + ': ' + ', '.join(missing)
    )

print('Using analysis directory:', ANALYSIS_DIR)


In [ ]:
summary = json.loads((ANALYSIS_DIR / 'summary.json').read_text(encoding='utf-8'))
model_scores = pd.read_csv(ANALYSIS_DIR / 'model_scores.csv')
sentence_summary = pd.read_csv(ANALYSIS_DIR / 'sentence_summary.csv')
group_comparison = pd.read_csv(ANALYSIS_DIR / 'stable_vs_unstable_comparison.csv')
top_unstable = pd.read_csv(ANALYSIS_DIR / 'top_unstable_sentences.csv')


## Headline Results

In [ ]:
headline = pd.DataFrame([
    {
        'runs': summary['num_runs'],
        'validation_examples': summary['num_validation_examples'],
        'non_unanimous_rate': round(summary['non_unanimous_rate'], 4),
        'accuracy_mean': round(summary['accuracy_mean'], 4),
        'accuracy_min': round(summary['accuracy_min'], 4),
        'accuracy_max': round(summary['accuracy_max'], 4),
        'accuracy_std': round(summary['accuracy_std'], 4),
        'mean_pairwise_agreement': round(summary['mean_pairwise_agreement'], 4),
        'agreement_accuracy_r': round(summary['agreement_accuracy_correlation']['pearson_r'], 4),
    }
])
headline


In [ ]:
best = summary['best_model_by_agreement']
display(Markdown(
    f"""
**Best model by agreement:** `{best['run_name']}`  
**Agreement score:** {best['agreement_score']:.4f}  
**Validation accuracy:** {best['eval_accuracy']:.4f}  
**Top-3 by validation accuracy:** {best['is_top_3_by_accuracy']}
"""
))


## Report-Ready Summary Text

In [ ]:
best = summary['best_model_by_agreement']
corr = summary['agreement_accuracy_correlation']

report_text = f"""
Across {summary['num_runs']} DistilBERT fine-tuning runs on SST-2 with fixed initialization and hyperparameters,
but different training-example permutations, {summary['non_unanimous_rate']:.2%} of validation sentences produced
non-unanimous predictions. Validation accuracy ranged from {summary['accuracy_min']:.4f} to {summary['accuracy_max']:.4f}
with mean {summary['accuracy_mean']:.4f} and standard deviation {summary['accuracy_std']:.4f}. The mean pairwise
agreement across runs was {summary['mean_pairwise_agreement']:.4f}. The model with the highest majority-agreement
stability score was {best['run_name']}, with agreement score {best['agreement_score']:.4f} and validation accuracy
{best['eval_accuracy']:.4f}. The Pearson correlation between agreement score and validation accuracy was
{corr['pearson_r']:.4f} (p={corr['p_value']:.4g}).
""".strip()

display(Markdown(report_text.replace('\n', ' ')))
print(report_text)


## Model Ranking Table

In [ ]:
ranking = model_scores.copy()
ranking['agreement_rank'] = ranking['agreement_score'].rank(ascending=False, method='min').astype(int)
ranking['accuracy_rank'] = ranking['eval_accuracy'].rank(ascending=False, method='min').astype(int)
ranking = ranking.sort_values(['agreement_rank', 'accuracy_rank'])

ranking[[
    'run_name',
    'agreement_rank',
    'agreement_score',
    'agreement_score_excluding_ties',
    'eval_accuracy',
    'accuracy_rank',
    'tie_rate',
    'permutation_seed',
]]


## Stable vs Unstable Inputs

In [ ]:
group_comparison


In [ ]:
stable_count = int((sentence_summary['disagreement_count'] == 0).sum())
unstable_count = int((sentence_summary['disagreement_count'] > 0).sum())
max_disagreement = int(sentence_summary['disagreement_count'].max())

display(Markdown(
    f"""
**Stable sentences:** {stable_count}  
**Non-unanimous sentences:** {unstable_count}  
**Maximum disagreement count:** {max_disagreement}
"""
))


## Most Ordering-Sensitive Sentences

In [ ]:
top_unstable[[
    'example_id',
    'sentence',
    'label',
    'majority_label',
    'disagreement_count',
    'mean_confidence',
    'confidence_std',
    'token_length',
    'has_negation',
    'has_contrast',
]].head(20)


## Figures

In [ ]:
for plot_name in [
    'disagreement_histogram.png',
    'pairwise_agreement_heatmap.png',
    'confidence_variance_histogram.png',
    'agreement_vs_accuracy.png',
]:
    display(Markdown(f'### {plot_name}'))
    display(Image(filename=str(ANALYSIS_DIR / plot_name)))


## Optional Export Cells

Run these if you want cleaner CSV files for the paper tables.

In [ ]:
paper_tables_dir = ANALYSIS_DIR / 'paper_tables'
paper_tables_dir.mkdir(exist_ok=True)

headline.to_csv(paper_tables_dir / 'headline_metrics.csv', index=False)
ranking.to_csv(paper_tables_dir / 'model_ranking.csv', index=False)
group_comparison.to_csv(paper_tables_dir / 'stable_vs_unstable.csv', index=False)
top_unstable.head(20).to_csv(paper_tables_dir / 'top20_unstable_sentences.csv', index=False)

print('Wrote report tables to', paper_tables_dir)
